# 🧑‍💼 AI-Based Employee Attrition Prediction
### 🚀 End-to-End Classification Project

**Goal:** Predict whether an employee is likely to **Stay 🟢** or **Leave 🔴**.

`Load Data → EDA → Clean → Preprocess → Train → Evaluate → Save Model`

## 🎯 Project Objective

Use employee information such as **age, income, overtime, job satisfaction, role and experience** to build a Machine Learning classification system.

### Expected Output
**Stay 🟢 | Leave 🔴 | Attrition Probability 📊**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
print('✅ Libraries loaded')

## 📂 1. Load Dataset

Place your file at: `data/employee_attrition.csv`

In [ ]:
df = pd.read_csv('../data/employee_attrition.csv')
print(f'✅ Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]:,} columns')
display(df.head())

## 🔎 2. Understand the Data

In [ ]:
print('📐 Shape:', df.shape)
print('\n📋 Columns:')
print(df.columns.tolist())
print('\n🧾 Data Types:')
display(df.dtypes.to_frame('Type'))

In [ ]:
print('❓ Missing Values')
display(df.isnull().sum().sort_values(ascending=False).to_frame('Missing'))
print('♻️ Duplicate rows:', df.duplicated().sum())

## 🎯 3. Target — Attrition

In [ ]:
TARGET = 'Attrition'
if TARGET not in df.columns:
    raise KeyError(f'{TARGET} not found. Available columns: {df.columns.tolist()}')

print('Target distribution:')
display(df[TARGET].value_counts())

plt.figure(figsize=(7,5))
sns.countplot(data=df, x=TARGET)
plt.title('🧑‍💼 Employee Attrition Distribution')
plt.show()

## 📊 4. Exploratory Data Analysis

### Questions
- Does overtime relate to attrition?
- Does income differ between groups?
- Which job roles show higher attrition?
- Does satisfaction relate to leaving?

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(include='object').columns.tolist()
print('🔢 Numerical:', numeric_cols)
print('\n🔤 Categorical:', categorical_cols)

In [ ]:
for col in [c for c in categorical_cols if c != TARGET][:4]:
    plt.figure(figsize=(9,5))
    sns.countplot(data=df, x=col, hue=TARGET)
    plt.title(f'{col} vs Attrition')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
for col in numeric_cols[:6]:
    plt.figure(figsize=(8,5))
    sns.boxplot(data=df, x=TARGET, y=col)
    plt.title(f'{col} vs Attrition')
    plt.tight_layout()
    plt.show()

## 🧠 5. Prepare Features

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

X = df.drop(columns=[TARGET])
y = df[TARGET].copy()

if y.dtype == 'object':
    vals = set(y.dropna().astype(str).str.lower().str.strip().unique())
    if vals <= {'yes','no'}:
        y = y.astype(str).str.lower().str.strip().map({'no':0,'yes':1})
    else:
        y = LabelEncoder().fit_transform(y.astype(str))

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()
print('Numeric features:', len(numeric_features))
print('Categorical features:', len(categorical_features))

## ✂️ 6. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print('Training:', X_train.shape)
print('Testing :', X_test.shape)

## ⚙️ 7. Preprocessing Pipeline

In [ ]:
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])
print('✅ Preprocessing ready')

## 🤖 8. Train Multiple Classification Models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

models = {
    'Logistic Regression': LogisticRegression(max_iter=2000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
}

results=[]
trained={}
for name, model in models.items():
    pipe=Pipeline([('preprocessor',preprocessor),('model',model)])
    pipe.fit(X_train,y_train)
    pred=pipe.predict(X_test)
    prob=pipe.predict_proba(X_test)[:,1]
    results.append([name, accuracy_score(y_test,pred), precision_score(y_test,pred,zero_division=0), recall_score(y_test,pred,zero_division=0), f1_score(y_test,pred,zero_division=0), roc_auc_score(y_test,prob)])
    trained[name]=pipe

results_df=pd.DataFrame(results,columns=['Model','Accuracy','Precision','Recall','F1','ROC-AUC']).sort_values('F1',ascending=False)
display(results_df.style.format({c:'{:.3f}' for c in results_df.columns[1:]}))

## 🏆 9. Best Model Evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

best_name=results_df.iloc[0]['Model']
best_model=trained[best_name]
best_pred=best_model.predict(X_test)

print('🏆 Selected model:', best_name)
print('\nClassification Report:\n')
print(classification_report(y_test,best_pred,zero_division=0))

cm=confusion_matrix(y_test,best_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues')
plt.title(f'Confusion Matrix — {best_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 💡 10. Project Insights

After execution, note:

- 📌 Dataset size
- 📌 Attrition distribution
- 📌 Important EDA findings
- 📌 Best model metrics
- 📌 Important employee factors

### 🚀 Final Stage
**Save Model → Streamlit Dashboard → Employee Risk Prediction**

In [ ]:
import joblib
from pathlib import Path
Path('../models').mkdir(exist_ok=True)
joblib.dump(best_model,'../models/best_model.pkl')
print('✅ Saved: models/best_model.pkl')